# 08 — Interactive Dashboard: Data Pipeline
**Finding Hidden Gems: 25 Years of WNBA Draft Value and Player Success**

The dashboard itself (`dashboard/index.html`) is a self-contained, single-file HTML/CSS/JS
app — open it directly in a browser, no server needed. It reads from `dashboard/data.js`,
a trimmed JSON bundle assembled from every CSV this project has produced so far.

This notebook documents (and reproduces) exactly how that bundle is built, so the
dashboard can be regenerated any time the upstream analysis changes.

**Pages:** Overview (Draft Value Calculator + GM Simulator), Draft Explorer (search/filter/
sort across all 1,064 picks — doubles as Player Search), Pick Value (the full expected-value,
bust-rate, and star-rate curves), Team Rankings, School Rankings, Hidden Gems, Busts, and
Timeline (draft class strength by year).


In [1]:
import pandas as pd
import json
import numpy as np

pd.set_option('display.max_columns', None)

clean = pd.read_csv('../data/wnba_draft_clean.csv')
steals = pd.read_csv('../data/draft_steal_scores.csv')
pick_value = pd.read_csv('../data/draft_pick_value.csv')
franchise = pd.read_csv('../data/franchise_draft_scorecard.csv')

steals['games_ev'] = clean['games'].fillna(0)
print(f"Loaded {len(clean)} picks, {len(pick_value)} pick-value rows, {len(franchise)} franchises")


Loaded 1064 picks, 64 pick-value rows, 18 franchises


## Trim to what the dashboard actually renders

No need to ship every column - the dashboard's tables and charts use a fixed subset.

In [2]:
def clean_records(df, cols):
    d = df[cols].copy().replace({np.nan: None})
    return json.loads(d.to_json(orient='records'))

picks = clean_records(steals, [
    'overall_pick', 'year', 'round', 'team', 'franchise', 'player', 'college_display',
    'is_international', 'years_played', 'win_shares_ev', 'games_ev', 'expected_win_shares',
    'steal_score', 'too_early_to_judge',
])
pick_curve = clean_records(pick_value, [
    'overall_pick', 'n', 'avg_win_shares', 'expected_win_shares', 'bust_rate', 'star_rate',
])
franchise_records = clean_records(franchise, [
    'franchise', 'picks_judged', 'avg_win_shares', 'avg_career_games', 'avg_steal_score',
    'steal_rate', 'bust_rate', 'total_win_shares', 'total_expected_win_shares', 'draft_roi',
    'total_picks_alltime',
])
print(f"{len(picks)} picks, {len(pick_curve)} curve points, {len(franchise_records)} franchises trimmed for the frontend")


1064 picks, 64 curve points, 18 franchises trimmed for the frontend


## College rankings, computed fresh for the dashboard

In [3]:
judgeable = steals[~steals['too_early_to_judge']]
us_judgeable = judgeable[judgeable['college_display'].notna()]
college_agg = us_judgeable.groupby('college_display').agg(
    picks=('player', 'count'),
    avg_win_shares=('win_shares_ev', 'mean'),
    total_win_shares=('win_shares_ev', 'sum'),
    avg_steal_score=('steal_score', 'mean'),
).reset_index()
all_time_counts = clean.groupby('college_display').size().rename('total_picks_alltime').reset_index()
college_agg = college_agg.merge(all_time_counts, on='college_display', how='left').sort_values('total_win_shares', ascending=False)
colleges = clean_records(college_agg, ['college_display','picks','avg_win_shares','total_win_shares','avg_steal_score','total_picks_alltime'])
print(f"{len(colleges)} colleges in the dashboard bundle")


161 colleges in the dashboard bundle


## Pull in the Hidden Gems tables and model results from earlier notebooks

In [4]:
def load_csv_records(path):
    d = pd.read_csv(f'../data/{path}').replace({np.nan: None})
    return json.loads(d.to_json(orient='records'))

gems = {
    'best_steals': load_csv_records('gems_best_steals.csv'),
    'class_strength': load_csv_records('gems_class_strength.csv'),
    'second_round': load_csv_records('gems_second_round.csv'),
    'overlooked_colleges': load_csv_records('gems_overlooked_colleges.csv'),
    'international': load_csv_records('gems_international.csv'),
}
biggest_busts = clean_records(
    judgeable.nsmallest(15, 'steal_score'),
    ['player','year','overall_pick','round','team','college_display','win_shares_ev','expected_win_shares','steal_score']
)
feature_importance = load_csv_records('feature_importance.csv')
model_comparison = json.loads(pd.read_csv('../data/model_comparison.csv').to_json(orient='records'))
print("Gems tables:", {k: len(v) for k, v in gems.items()})


Gems tables: {'best_steals': 15, 'class_strength': 24, 'second_round': 12, 'overlooked_colleges': 12, 'international': 12}


## Write the bundle

In [5]:
bundle = {
    'picks': picks,
    'pick_curve': pick_curve,
    'franchise': franchise_records,
    'colleges': colleges,
    'gems': gems,
    'biggest_busts': biggest_busts,
    'feature_importance': feature_importance,
    'model_comparison': model_comparison,
    'meta': {
        'total_picks': len(clean),
        'years': [int(clean.year.min()), int(clean.year.max())],
        'star_threshold': 20.0,
    },
}

out_path = '../dashboard/data.js'
with open(out_path, 'w') as f:
    f.write('const DASHBOARD_DATA = ')
    json.dump(bundle, f, separators=(',', ':'))
    f.write(';')

import os
print(f"Wrote {out_path} ({os.path.getsize(out_path):,} bytes)")


Wrote ../dashboard/data.js (393,297 bytes)


## Verification

Every render path in `dashboard/index.html` was tested against this exact bundle using a
headless-browser smoke test (Playwright): tab switching, the Draft Value Calculator slider,
the GM Simulator lookup, Draft Explorer search/sort/filter/pagination, and every chart
(pick value curve, bust/star rate curves, team bars, school bars, timeline bars) were all
exercised with zero console errors, and spot-checked against these notebooks' own outputs
(e.g. Tamika Catchings' +76.2 steal score, Connecticut Sun's 2.88 average team steal score).
Mobile layout (390px viewport) was checked separately for horizontal overflow.
